# 🧹 preparar_base_datos — Limpieza + Balanceo (estructura por clases)

**Salida en `BaseDatos/`:** 8 carpetas (una por clase), cada una con 4 CSV (uno por herramienta).
Cada CSV: 3.481 filas (Web es la minoritaria) + columnas limpias.

**Filtros aplicados:**
- Identificadoras (IPs, puertos, timestamps) → fuera (sesgo).
- Auxiliares (`subtipo_dos/ddos`, `archivo_origen`) → fuera.
- 9 columnas con valor 0 del TFG (solo CIC).
- Columnas QUIC de Nemea → fuera (Nemea homogéneo en todas las clases).

**Reproducibilidad:** `random_state=42`.

In [12]:
# ============================================================
# CELDA 0 — IMPORTS Y RUTAS
# ============================================================
import pandas as pd
from pathlib import Path
import gc
import warnings
warnings.filterwarnings('ignore')

BASE_IN   = Path('/home/miguel/Escritorio/TFM/TFM_Miguel/MuestrasComunes')
BASE_OUT  = Path('/home/miguel/Escritorio/TFM/TFM_Miguel/BaseDatos')
BASE_OUT.mkdir(parents=True, exist_ok=True)

CLASES = ['BenignTraffic', 'DictionaryBruteForce', 'Recon', 'Mirai',
          'Spoofing', 'DoS', 'DDoS', 'Web']
HERRAMIENTAS = ['CIC', 'Nemea', 'Tstat', 'Tshark']

N_MUESTRAS = 3481
SEED = 42

print(f'Entrada:  {BASE_IN}')
print(f'Salida:   {BASE_OUT}')
print(f'Clases:   {len(CLASES)}')
print(f'Muestras por clase: {N_MUESTRAS:,}')

Entrada:  /home/miguel/Escritorio/TFM/TFM_Miguel/MuestrasComunes
Salida:   /home/miguel/Escritorio/TFM/TFM_Miguel/BaseDatos
Clases:   8
Muestras por clase: 3,481


In [13]:
# ============================================================
# CELDA 1 — COLUMNAS A ELIMINAR
# ============================================================

COLS_RUIDO_COMUNES = ['subtipo_dos', 'subtipo_ddos',
                       'archivo_origen', 'archivo_origen_nemea']

COLS_DROP = {
    'CIC': [
        'Flow ID', 'Src IP', 'Dst IP', 'Src Port', 'Dst Port', 'Timestamp',
        'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags',
        'URG Flag Cnt', 'CWR Flag Count', 'ECE Flag Cnt',
        'Fwd Bytes/Bulk Avg', 'Fwd Bulk Rate Avg', 'Fwd Packet/Bulk Avg',
    ],
    'Nemea': [
        'ipaddr SRC_IP', 'ipaddr DST_IP',
        'uint16 SRC_PORT', 'uint16 DST_PORT',
        'time TIME_FIRST', 'time TIME_LAST',
    ],
    'Tstat': [
        'c_ip', 's_ip', 'c_port', 's_port',
        'first', 'c_first_abs',
    ],
    'Tshark': [
        'src_ip', 'dst_ip', 'src_port', 'dst_port',
        'start_epoch',
    ],
}

for h in COLS_DROP:
    COLS_DROP[h] = list(set(COLS_DROP[h] + COLS_RUIDO_COMUNES))

print('Columnas a eliminar por herramienta:')
for h, c in COLS_DROP.items():
    print(f'  {h:8s}: {len(c)} columnas')
print(f'\n  + columnas con "QUIC" en el nombre (afecta a Nemea, ~14 cols)')

Columnas a eliminar por herramienta:
  CIC     : 19 columnas
  Nemea   : 10 columnas
  Tstat   : 10 columnas
  Tshark  : 9 columnas

  + columnas con "QUIC" en el nombre (afecta a Nemea, ~14 cols)


In [14]:
# ============================================================
# CELDA 2 — FUNCIÓN AUXILIAR
# ============================================================

def procesar_csv(ruta_csv, herramienta, n_muestras, seed):
    df = pd.read_csv(ruta_csv, low_memory=False)
    df = df.drop(columns=COLS_DROP[herramienta], errors='ignore')
    cols_quic = [c for c in df.columns if 'QUIC' in c]
    if cols_quic:
        df = df.drop(columns=cols_quic)
    if len(df) > n_muestras:
        df = df.sample(n=n_muestras, random_state=seed).reset_index(drop=True)
    elif len(df) < n_muestras:
        print(f'    ⚠ {ruta_csv.name}: solo {len(df)} filas (<{n_muestras}), se mantienen todas')
        df = df.reset_index(drop=True)
    else:
        df = df.reset_index(drop=True)
    return df

print('✓ Función procesar_csv lista')

✓ Función procesar_csv lista


In [15]:
# ============================================================
# CELDA 3 — PROCESADO PRINCIPAL
# 8 carpetas en BaseDatos/, cada una con 4 CSV (uno por herramienta)
# ============================================================

resumen = {}

for clase in CLASES:
    print(f'\n{"="*60}')
    print(f'  CLASE: {clase}')
    print(f'{"="*60}')

    carpeta_out = BASE_OUT / clase
    carpeta_out.mkdir(parents=True, exist_ok=True)
    resumen[clase] = {}

    for herr in HERRAMIENTAS:
        ruta_in = BASE_IN / clase / f'{herr}_comunes.csv'
        if not ruta_in.exists():
            print(f'  ✗ NO existe {ruta_in}')
            continue
        df = procesar_csv(ruta_in, herr, N_MUESTRAS, SEED)
        ruta_out = carpeta_out / f'{herr}.csv'
        df.to_csv(ruta_out, index=False)
        resumen[clase][herr] = (len(df), df.shape[1])
        print(f'  ✓ {herr:7s} → {len(df):>5,} filas x {df.shape[1]:>3} cols  → {ruta_out.name}')
        del df
        gc.collect()


  CLASE: BenignTraffic
  ✓ CIC     → 3,481 filas x  71 cols  → CIC.csv
  ✓ Nemea   → 3,481 filas x 187 cols  → Nemea.csv
  ✓ Tstat   → 3,481 filas x 132 cols  → Tstat.csv
  ✓ Tshark  → 3,481 filas x  58 cols  → Tshark.csv

  CLASE: DictionaryBruteForce
  ✓ CIC     → 3,481 filas x  71 cols  → CIC.csv
  ✓ Nemea   → 3,481 filas x 187 cols  → Nemea.csv
  ✓ Tstat   → 3,481 filas x 132 cols  → Tstat.csv
  ✓ Tshark  → 3,481 filas x  58 cols  → Tshark.csv

  CLASE: Recon
  ✓ CIC     → 3,481 filas x  71 cols  → CIC.csv
  ✓ Nemea   → 3,481 filas x 187 cols  → Nemea.csv
  ✓ Tstat   → 3,481 filas x 132 cols  → Tstat.csv
  ✓ Tshark  → 3,481 filas x  58 cols  → Tshark.csv

  CLASE: Mirai
  ✓ CIC     → 3,481 filas x  71 cols  → CIC.csv
  ✓ Nemea   → 3,481 filas x 187 cols  → Nemea.csv
  ✓ Tstat   → 3,481 filas x 132 cols  → Tstat.csv
  ✓ Tshark  → 3,481 filas x  58 cols  → Tshark.csv

  CLASE: Spoofing
  ✓ CIC     → 3,481 filas x  71 cols  → CIC.csv
  ✓ Nemea   → 3,481 filas x 187 cols  → Nemea.csv


In [16]:
# ============================================================
# CELDA 4 — VERIFICACIÓN FINAL
# ============================================================

print('='*70)
print(f'  RESUMEN — BaseDatos/')
print('='*70)
print(f"\n  {'Clase':25s} {'CIC':>12s} {'Nemea':>14s} {'Tstat':>12s} {'Tshark':>12s}")
print('  ' + '-'*78)
for clase, datos in resumen.items():
    fila = f'  {clase:25s}'
    for herr in HERRAMIENTAS:
        if herr in datos:
            f, c = datos[herr]
            fila += f' {f:>5,}x{c:>3}  '
        else:
            fila += f' {"-":>11s}  '
    print(fila)

print('\n' + '='*70)
print('  CHEQUEO DE CONSISTENCIA (mismas columnas en todas las clases)')
print('='*70)
for herr in HERRAMIENTAS:
    cols_por_clase = {clase: datos[herr][1] for clase, datos in resumen.items() if herr in datos}
    valores = set(cols_por_clase.values())
    if len(valores) == 1:
        print(f'  ✓ {herr:8s}: TODAS las clases con {valores.pop()} columnas')
    else:
        print(f'  ⚠ {herr:8s}: columnas distintas → {cols_por_clase}')

print(f'\n  ✓ Listo.')

  RESUMEN — BaseDatos/

  Clase                              CIC          Nemea        Tstat       Tshark
  ------------------------------------------------------------------------------
  BenignTraffic             3,481x 71   3,481x187   3,481x132   3,481x 58  
  DictionaryBruteForce      3,481x 71   3,481x187   3,481x132   3,481x 58  
  Recon                     3,481x 71   3,481x187   3,481x132   3,481x 58  
  Mirai                     3,481x 71   3,481x187   3,481x132   3,481x 58  
  Spoofing                  3,481x 71   3,481x187   3,481x132   3,481x 58  
  DoS                       3,481x 71   3,481x187   3,481x132   3,481x 58  
  DDoS                      3,481x 71   3,481x187   3,481x132   3,481x 58  
  Web                       3,481x 71   3,481x187   3,481x132   3,481x 58  

  CHEQUEO DE CONSISTENCIA (mismas columnas en todas las clases)
  ✓ CIC     : TODAS las clases con 71 columnas
  ✓ Nemea   : TODAS las clases con 187 columnas
  ✓ Tstat   : TODAS las clases con 132 column